# Data Preparation for Kaya ChatBot

This notebook:
1. Loads configuration from config.yaml
2. Reads and cleans WhatsApp export
3. Creates token-based chunks with proper Llama-3 formatting
4. Splits data into train/validation sets
5. Saves processed datasets


In [10]:
import sys
import os
import json
import yaml
from datasets import Dataset
from sklearn.model_selection import train_test_split

# Add src to path so we can import kaya_chatbot
sys.path.append(os.path.abspath(os.path.join('..', '..', 'src')))

from kaya_chatbot.data import WhatsAppReader, ConversationFormatter

In [4]:
# 1. Load Configuration
with open(r"C:\Users\guga\Desktop\KayaChatBot\config.yaml", 'r', encoding='utf-8') as f:
    config = yaml.safe_load(f)

# Extract settings - fix file path encoding
INPUT_FILE = r"C:\Users\guga\Desktop\KayaChatBot\data\wpp\WhatsApp Chat with Kaya 👀 - gugu.txt"
OUTPUT_DIR = os.path.dirname(config['data']['output_file'])
SYSTEM_PROMPT = config['data']['system_prompt']
MAX_TOKENS = config['data']['max_tokens_per_chunk']
OVERLAP_TOKENS = config['data']['overlap_tokens']
TRAIN_TEST_SPLIT = config['data']['train_test_split']

print(f"✓ Configuration loaded")
print(f"  Input file: {INPUT_FILE}")
print(f"  Max tokens per chunk: {MAX_TOKENS}")
print(f"  Train/test split: {TRAIN_TEST_SPLIT}")

✓ Configuration loaded
  Input file: C:\Users\guga\Desktop\KayaChatBot\data\wpp\WhatsApp Chat with Kaya 👀 - gugu.txt
  Max tokens per chunk: 2048
  Train/test split: 0.9


In [5]:
# 2. Read WhatsApp Data
reader = WhatsAppReader(INPUT_FILE)
messages = reader.read()
print(f"✓ Total messages loaded: {len(messages)}")

# Show sample
print("\n--- Sample Messages ---")
for msg in messages[:5]:
    print(f"{msg.date} - {msg.sender}: {msg.content[:100]}...")

✓ Total messages loaded: 17120

--- Sample Messages ---
3/26/20, 15:29 - Gil João: Listen to this song with headphones (put on the 2 headphones). It is the new music of the Pentatonix...
3/26/20, 15:29 - Gil João: Oiçam com phones please...
3/26/20, 15:29 - Gil João: Tem mm de ser com phones...
3/26/20, 15:33 - Gil João: Fechem os olhos neste Ahahaha...
3/26/20, 15:34 - Peter: Parece-me normal, apenas bem masterizada ahah...


In [6]:
# 3. Format Data into Token-Based Chunks
formatter = ConversationFormatter(messages)
dataset = formatter.to_instruction_chunks(
    system_prompt=SYSTEM_PROMPT,
    max_tokens=MAX_TOKENS,
    overlap_tokens=OVERLAP_TOKENS
)
print(f"✓ Total training chunks created: {len(dataset)}")

# Show sample chunk
print("\n--- Sample Chunk ---")
print(f"System: {dataset[0]['system'][:100]}...")
print(f"Text preview: {dataset[0]['text'][:300]}...")

c:\Users\guga\Desktop\KayaChatBot\kaya_chatbot_env\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\guga\.cache\huggingface\hub\models--unsloth--llama-3-8b-Instruct-bnb-4bit. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Created 169 token-based chunks from 17120 messages.
✓ Total training chunks created: 169

--- Sample Chunk ---
System: You are Kaya, an AI member of this group chat who has read all the conversation history. You can ans...
Text preview: Gil João: Listen to this song with headphones (put on the 2 headphones). It is the new music of the Pentatonix, composed with 8D technology. Listen to it only with headphones. It will be the first time that you will listen to that song with your brain and not with your ears. You will feel the music ...


In [7]:
# 4. Format with Llama-3 Chat Template
formatted_chunks = []
for entry in dataset:
    formatted_text = formatter.format_for_llama3(entry)
    formatted_chunks.append({
        'formatted_text': formatted_text,
        'system': entry['system'],
        'text': entry['text']
    })

print(f"✓ Formatted {len(formatted_chunks)} chunks with Llama-3 template")
print(f"\n--- Sample Formatted Chunk ---")
print(formatted_chunks[0]['formatted_text'][:500] + "...")

✓ Formatted 169 chunks with Llama-3 template

--- Sample Formatted Chunk ---
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are Kaya, an AI member of this group chat who has read all the conversation history. You can answer questions about what people said, their interests, inside jokes, and group dynamics. Always be helpful and accurate based on the chat history.<|eot_id|><|start_header_id|>user<|end_header_id|>

Remember this conversation history.<|eot_id|><|start_header_id|>assistant<|end_header_id|>

Gil João: Listen to this song with headphones (pu...


In [8]:
# 5. Split into Train/Validation
train_data, val_data = train_test_split(
    formatted_chunks, 
    train_size=TRAIN_TEST_SPLIT, 
    random_state=42,
    shuffle=True
)

print(f"✓ Train samples: {len(train_data)}")
print(f"✓ Validation samples: {len(val_data)}")

# Convert to Hugging Face datasets
train_dataset = Dataset.from_list(train_data)
val_dataset = Dataset.from_list(val_data)

print(f"\n✓ Datasets created successfully")

✓ Train samples: 152
✓ Validation samples: 17

✓ Datasets created successfully


In [9]:
# 6. Save Datasets
os.makedirs(OUTPUT_DIR, exist_ok=True)
train_file = os.path.join(OUTPUT_DIR, "train_data.jsonl")
val_file = os.path.join(OUTPUT_DIR, "val_data.jsonl")

# Save as JSONL
with open(train_file, 'w', encoding='utf-8') as f:
    for entry in train_data:
        json.dump(entry, f, ensure_ascii=False)
        f.write('\n')

with open(val_file, 'w', encoding='utf-8') as f:
    for entry in val_data:
        json.dump(entry, f, ensure_ascii=False)
        f.write('\n')

print(f"💾 Datasets saved:")
print(f"   • Train: {train_file}")
print(f"   • Validation: {val_file}")
print(f"\n✅ Data preparation complete!")

💾 Datasets saved:
   • Train: C:\Users\guga\Desktop\KayaChatBot\data\train_data.jsonl
   • Validation: C:\Users\guga\Desktop\KayaChatBot\data\val_data.jsonl

✅ Data preparation complete!
